# Estadísticas por cliente — corrida diaria

Calcula las estadísticas de cada combinación de categorías y **reemplaza completa** la tabla destino.
Toda la configuración está en **`st_oracle.py`** y todo el cálculo en **`stats_engine.py`**; este
notebook no tiene nada configurable salvo la celda de parámetros.

```
 ORIGEN (lector)                      MOTOR                        DESTINO (escritor)
 SQL_FUENTE  [:desde, :hasta)  ─┐
                                ├─> historia ─> StatsEngine.run() ─> TABLA_DESTINO
 BD_HISTORIAL guardado  < desde ─┘   diaria      85 métricas          (se reemplaza)
 (sólo en la incremental)                         + Pareto/NBD
```

### Nodo en el pipeline (Elyra / OpenShift AI)

| Propiedad | Valor |
|---|---|
| **File Dependencies** | `st_oracle.py`, `stats_engine.py` |
| **Environment Variables** | `ORA_USER`, `ORA_PASSWORD`, `ORA_DSN`, `ORA_DEST_USER`, `ORA_DEST_PASSWORD`, `ORA_DEST_DSN` |
| **Opcionales** | `ST_MODO`, `ST_RELEER_MESES`, `ST_RELEER_DESDE`, `ST_FECHA_EJECUCION`, `ST_DRY_RUN` |
| **Instalación** | `pip install pandas numpy scipy oracledb` |

### Modos de corrida

| `ST_MODO` | Qué hace |
|---|---|
| `auto` (default) | incremental si la tabla destino está sana; si no, completa y dice por qué |
| `completo` | relee toda la fuente y recalcula todo. **Es la forma de recalcular todo cuando quieras** |
| `incremental` | exige incremental: si la tabla no sirve, corta con error en vez de pasar a completa |

Para corregir algo más viejo que la ventana sin releer todo: `ST_RELEER_DESDE=2025-01-01`.

In [ ]:
# Parámetros (tag `parameters`)
import os

MODO = os.getenv("ST_MODO") or None                    # auto | completo | incremental
RELEER_MESES = os.getenv("ST_RELEER_MESES") or None    # meses cerrados a releer
RELEER_DESDE = os.getenv("ST_RELEER_DESDE") or None    # "2025-01-01": corregir desde esa fecha
FECHA_EJECUCION = os.getenv("ST_FECHA_EJECUCION") or None
DRY_RUN = os.getenv("ST_DRY_RUN", "0") == "1"
print(f"MODO={MODO!r} RELEER_MESES={RELEER_MESES!r} RELEER_DESDE={RELEER_DESDE!r} "
      f"FECHA_EJECUCION={FECHA_EJECUCION!r} DRY_RUN={DRY_RUN}")

## 1. Setup

In [ ]:
import sys, time
sys.path.insert(0, os.getcwd())

import pandas as pd
import st_oracle as io
from stats_engine import StatsEngine

log = io.configurar_logging("estadisticas")
cfg = io.build_config(fecha_ejecucion=FECHA_EJECUCION)
motor = StatsEngine(cfg)
f = motor.fechas
log.info("ejecución %s | último día incluido %s | %d métricas | modelo de actividad %s",
         f.hoy.date(), f.ayer.date(), len(motor.metricas), cfg.modelo_actividad)
t_inicio = time.time()

## 2. Tabla destino

Imprime el `CREATE TABLE` exacto (sin constraints, con la descripción de cada columna) y valida
contra el diccionario de datos que no falte ninguna, **antes** de leer nada.

In [ ]:
print(io.ddl_sugerido(cfg))

with io.conexion_destino() as conn:
    io.validar_tabla(conn, cfg)

## 3. Plan: ¿completa o incremental?

Antes de decidir, valida la tabla destino con **una consulta agregada, sin traer los CLOB**:

| Chequeo | Si falla |
|---|---|
| la tabla tiene filas | completa |
| todas las filas tienen la **misma** `FECHA_CORTE` | completa: la carga anterior quedó mezclada |
| se sabe desde cuándo hay datos (`FECHA_DATOS_DESDE` única) | completa |
| `HUELLA_CONFIG` igual a la actual (categorías, SQL de la fuente, historial) | completa: cambió la configuración |
| todas las filas tienen `BD_HISTORIAL` | completa |

Si todo está bien, relee la fuente desde el 1° del mes `RELEER_MESES` antes del mes en curso. **Si la
última carga quedó más atrás** —el pipeline no corrió unos días—, relee desde el día siguiente a esa
carga, así nunca quedan huecos.

In [ ]:
with io.conexion_destino() as conn:
    plan = io.planificar(conn, cfg,
                         modo=MODO or io.MODO_CORRIDA,
                         releer_meses=int(RELEER_MESES) if RELEER_MESES else io.RELEER_MESES,
                         releer_desde=RELEER_DESDE)
    estado = io.leer_estado(conn, cfg) if plan.tipo == "INCREMENTAL" else None
log.info("plan: %s", plan)

## 4. Fuente e historia

Lee la fuente desde `plan.releer_desde` y valida que esté al día (`MAX_DIAS_SIN_DATOS`): si la fuente
no se cargó, corta acá en vez de calcular como si nadie hubiera comprado.

En la incremental, la historia es **lo guardado antes de esa fecha + la fuente desde esa fecha**. Lo
releído reemplaza a lo guardado: las correcciones dentro de la ventana se aplican solas, incluidas
las bajas. La auditoría muestra qué cambió en el tramo que se volvió a leer.

In [ ]:
with io.conexion_origen() as conn:
    fuente = io.leer_fuente(conn, cfg, plan)
historia, auditoria = io.combinar(estado, fuente, cfg, plan)
auditoria

## 5. Cálculo

Aunque la lectura sea incremental, se recalculan **todos** los clientes: los días sin compra, las
ventanas móviles y la probabilidad de actividad cambian para todos cada día, compren o no.

In [ ]:
out = motor.run(historia)
out = io.anotar(out, cfg, plan, fuente)

## 6. Control

In [ ]:
io.resumen(motor, out)
out.drop(columns=["BD_HISTORIAL"]).head(3).T

In [ ]:
# Parámetros de actividad por segmento: cuál usó parámetros propios y cuál GLOBAL, y por qué.
# compra_cada_dias y vida_media_dias traducen los parámetros a días (sólo Pareto/NBD).
motor.actividad_segmentos()

In [ ]:
motor.tiempos(10)

## 7. Guardar

Con `MODO_CARGA = "delete"` borra e inserta en una sola transacción: si algo falla, la tabla queda como
estaba, y la próxima corrida relee desde donde había quedado.

In [ ]:
if DRY_RUN:
    log.warning("DRY_RUN: no se escribe nada en %s", io.TABLA_DESTINO)
else:
    with io.conexion_destino() as conn:
        io.guardar(conn, out, cfg)
log.info("corrida %s OK en %.1fs", plan.tipo, time.time() - t_inicio)

## Cómo funciona la probabilidad de actividad (Pareto/NBD)

### La idea

Nunca vemos el día en que un cliente se va: sólo vemos que deja de comprar. El modelo supone que cada
cliente tiene, sin que lo veamos, **dos relojes**:

1. **Mientras está activo, compra a su propio ritmo** λ. Hay clientes de todas las semanas y clientes
   de una vez al año; esos ritmos varían entre clientes según una distribución Gamma(`r`, `alpha`).
2. **En algún momento se va**, con su propia tasa de abandono μ. También varía entre clientes, según
   una Gamma(`s`, `beta`).

Con la historia de un cliente —**cuántas veces compró** (`x`), **cuándo fue la última** (`tx`) y
**hace cuánto lo conocemos** (`T`)— el modelo compara dos explicaciones de su silencio desde la
última compra:

- **A. Sigue activo** y simplemente no le tocó comprar todavía.
- **B. Se fue** en algún momento entre su última compra y hoy.

`P(activo)` es el peso de A frente a A + B:

$$P(\text{activo}) = \frac{1}{1 + \dfrac{s}{r+s+x}\,(\alpha+T)^{r+x}\,(\beta+T)^{s}\,A_0}$$

donde $A_0$ integra todos los momentos posibles en que pudo haberse ido, entre `tx` y `T`.

### Qué la mueve, en la práctica

- **El silencio se mide contra el propio ritmo del cliente.** Uno que compra todas las semanas y lleva
  3 meses sin comprar: probabilidad baja. Uno que compra una vez al año y lleva 3 meses: normal,
  probabilidad alta. El mismo silencio significa cosas distintas.
- **Si compró ayer, da prácticamente 1.**
- **Cuanto más largo el silencio, más baja.** Y baja más rápido cuanto más frecuente era el cliente.
- **Aprende de todo el panel.** Los 4 parámetros se estiman con todos los clientes: uno con poca
  historia "toma prestado" cuánto abandono es normal en la población. Por eso un cliente de **una sola
  compra no da 1**: el modelo sabe qué tan común es que alguien compre una vez y no vuelva.

Leé los parámetros así: `r/alpha` es la tasa media de compra por día y `s/beta` la tasa media de
abandono por día (su inversa da una idea de la vida media de un cliente, en días).

### Por segmento

Con un solo modelo, "cuánto abandono es normal" se aprende de todo el panel. Si mezclás, por ejemplo,
mayoristas que compran cada semana con minoristas que compran dos veces al año, el modelo aprende un
promedio que no describe a ninguno. Con `SEGMENTOS_ACTIVIDAD = ["BD_CANAL"]` en `st_oracle.py` se ajusta
**un modelo por cada valor** (o combinación de valores, si ponés varias categorías), y cada cliente se
evalúa contra lo normal de su segmento.

| Caso | Qué parámetros usa |
|---|---|
| `SEGMENTOS_ACTIVIDAD = []` | GLOBAL: un modelo para todo el panel |
| segmento con `MIN_CLIENTES_SEGMENTO` grupos o más | los propios del segmento |
| segmento más chico | GLOBAL: con pocos clientes los 4 parámetros no se estiman bien |
| ajuste del segmento en el borde (casi sin abandono visible, o todos abandonan) | GLOBAL |
| categoría vacía | su propio segmento `(sin dato)`, con las mismas reglas |

`BD_SEGMENTO_ACTIVIDAD` dice, en cada fila, con qué parámetros se calculó, y la tabla de la sección 6
muestra los de cada segmento. Las categorías del segmento tienen que estar en `CATEGORIAS`: para un
atributo del cliente usá una descripción `BD_` (valor más reciente); una clave `SK_`/`BK_` cambia el grano
de la tabla. Cambiar la segmentación **no** obliga a una corrida completa: el historial no depende de ella.

### Qué NO es

`P(activo)` **no es la probabilidad de que compre el mes que viene**. Un cliente anual puede estar
activo y no comprar en 6 meses. Es la probabilidad de que la relación siga viva.

### Por qué Pareto/NBD y no BG/NBD

BG/NBD sólo permite que el cliente se vaya **justo después de una compra**. Un cliente de una sola
compra nunca tuvo esa oportunidad, así que el modelo le asigna 100% de actividad aunque haya comprado
hace años. Pareto/NBD permite irse en cualquier momento. Comparados con datos reales y simulados:

| Prueba | BG/NBD | MBG/NBD | **Pareto/NBD** |
|---|---|---|---|
| CDNOW real, 23.570 clientes (60% de una compra): AUC al predecir quién vuelve | 0,320 | 0,700 | **0,722** |
| Simulación con estado real conocido: AUC | 0,950 | 0,976 | **0,978** |
| Simulación: error de calibración (Brier, menor es mejor) | 0,063 | 0,046 | **0,038** |

Los tres coinciden con la librería `lifetimes` (probabilidades con diferencia menor a 0,0001). Con
paneles grandes el ajuste usa una **muestra estable** de `max_clientes_ajuste` grupos —siempre los
mismos clientes, elegidos por un hash de sus claves—, reparte el cálculo entre los CPUs del pod, y la
probabilidad se calcula para todos. Con 100.000 clientes el ajuste tarda unos 3 segundos.

## Catálogo de columnas

Generado desde `stats_engine.catalogo()`. La tercera columna es la función a modificar.

| Columna | Descripción | Función |
|---|---|---|
| `MT_VENTA_YTD` | Venta del año en curso, desde el 1 de enero hasta ayer. USD. | `venta_ventana`(ventana='YTD') |
| `MT_VENTA_YTD_AP` | Año pasado de MT_VENTA_YTD: la misma ventana corrida un año atrás. USD. | `venta_ventana`(ventana='YTD', ap=True) |
| `MT_VENTA_YTD_MC` | Venta del año en curso a mes cerrado, desde el 1 de enero hasta el fin del último mes cerrado (en enero es 0). USD. | `venta_ventana`(ventana='YTD_MC') |
| `MT_VENTA_YTD_MC_AP` | Año pasado de MT_VENTA_YTD_MC: la misma ventana corrida un año atrás. USD. | `venta_ventana`(ventana='YTD_MC', ap=True) |
| `MT_VENTA_R12` | Venta de los últimos 365 días hasta ayer. USD. | `venta_ventana`(ventana='R12') |
| `MT_VENTA_R12_AP` | Año pasado de MT_VENTA_R12: la misma ventana corrida un año atrás. USD. | `venta_ventana`(ventana='R12', ap=True) |
| `MT_VENTA_R12_MC` | Venta de los últimos 12 meses cerrados completos. USD. | `venta_ventana`(ventana='R12_MC') |
| `MT_VENTA_R12_MC_AP` | Año pasado de MT_VENTA_R12_MC: la misma ventana corrida un año atrás. USD. | `venta_ventana`(ventana='R12_MC', ap=True) |
| `MT_VENTA_R6` | Venta de los últimos 183 días hasta ayer. USD. | `venta_ventana`(ventana='R6') |
| `MT_VENTA_R6_AP` | Año pasado de MT_VENTA_R6: la misma ventana corrida un año atrás. USD. | `venta_ventana`(ventana='R6', ap=True) |
| `MT_VENTA_R6_MC` | Venta de los últimos 6 meses cerrados completos. USD. | `venta_ventana`(ventana='R6_MC') |
| `MT_VENTA_R6_MC_AP` | Año pasado de MT_VENTA_R6_MC: la misma ventana corrida un año atrás. USD. | `venta_ventana`(ventana='R6_MC', ap=True) |
| `MT_MAXMONEY` | Máxima venta en un día de compra (suma del día). USD. | `max_money` |
| `MT_MINMONEY` | Mínima venta en un día de compra; negativa si hubo devolución neta. USD. | `min_money` |
| `MT_MAXDAYS` | Máximo de días sin comprar entre dos días de compra (dos días seguidos = 0). Nulo con un solo día de compra. | `max_days` |
| `MT_MINDAYS` | Mínimo de días sin comprar entre dos días de compra (dos días seguidos = 0). Nulo con un solo día de compra. | `min_days` |
| `MT_RANGODIAS` | Días entre el primer y el último día de compra. | `rango_dias` |
| `MT_DIASNCOMPRA` | Días desde el último día de compra hasta ayer (compró ayer = 0). | `dias_sin_compra` |
| `MT_IDDPORCENTUAL` | Pendiente de la recta de venta mensual (meses cerrados desde la primera compra, sin compra = 0), expresada en gradianes (0 plano, 100 creciente, -100 decreciente). Histórico. | `idd_porcentual` |
| `MT_IDDPORCENTUAL_R12` | Pendiente de la recta de venta mensual de los últimos 12 meses cerrados, expresada en gradianes (0 plano, 100 creciente, -100 decreciente). | `idd_porcentual`(ventana='R12') |
| `MT_IDDPORCENTUAL_R24` | Pendiente de la recta de venta mensual de los últimos 24 meses cerrados, expresada en gradianes (0 plano, 100 creciente, -100 decreciente). | `idd_porcentual`(ventana='R24') |
| `MT_IDDPORCENTUAL_R36` | Pendiente de la recta de venta mensual de los últimos 36 meses cerrados, expresada en gradianes (0 plano, 100 creciente, -100 decreciente). | `idd_porcentual`(ventana='R36') |
| `MT_IDDPENDIENTE` | Pendiente de la recta de venta mensual en USD por mes, toda la historia (meses cerrados, sin compra = 0). | `idd_pendiente` |
| `MT_MARGENBRUTO` | Margen bruto histórico en %: suma del margen / suma de la venta. | `margen_bruto` |
| `MT_MARGENBRUTO_R12` | Margen bruto en % de los últimos 365 días: suma del margen / suma de la venta. | `margen_bruto`(ventana='R12') |
| `MT_DIASCOMPRADOS` | Cantidad de días distintos con compra, incluidos días con devolución. | `dias_comprados` |
| `MT_VOLUMENCOMPRA` | Venta total de toda la historia. USD. | `volumen_compra` |
| `MT_FRECUENCIACOMPRA` | Cada cuántos días compra en promedio: promedio de días entre días de compra consecutivos (compra diaria = 1). Histórico. | `frecuencia_compra` |
| `MT_FRECUENCIACOMPRA_R12` | Cada cuántos días compra en promedio, contando sólo intervalos con ambos días dentro de los últimos 365 días. | `frecuencia_compra`(ventana='R12') |
| `MT_FRECUENCIACOMPRA_R24` | Cada cuántos días compra en promedio, contando sólo intervalos con ambos días dentro de los últimos 730 días. | `frecuencia_compra`(ventana='R24') |
| `MT_FRECUENCIACOMPRA_R36` | Cada cuántos días compra en promedio, contando sólo intervalos con ambos días dentro de los últimos 1095 días. | `frecuencia_compra`(ventana='R36') |
| `MT_TICKETPROMEDIO` | Venta promedio por día de compra (venta / días de compra). Histórico. USD. | `ticket_promedio` |
| `MT_TICKETPROMEDIO_R12` | Venta promedio por día de compra en los últimos 365 días. USD. | `ticket_promedio`(ventana='R12') |
| `MT_TICKETPROMEDIO_R24` | Venta promedio por día de compra en los últimos 730 días. USD. | `ticket_promedio`(ventana='R24') |
| `MT_TICKETPROMEDIO_R36` | Venta promedio por día de compra en los últimos 1095 días. USD. | `ticket_promedio`(ventana='R36') |
| `MT_FRECUENCIACOMPRA_STD` | Desvío estándar de los días entre días de compra consecutivos. Histórico. | `frecuencia_compra_std` |
| `MT_FRECUENCIACOMPRA_STD_R12` | Desvío estándar de los días entre compras, intervalos dentro de los últimos 365 días. | `frecuencia_compra_std`(ventana='R12') |
| `MT_FRECUENCIACOMPRA_STD_R24` | Desvío estándar de los días entre compras, intervalos dentro de los últimos 730 días. | `frecuencia_compra_std`(ventana='R24') |
| `MT_FRECUENCIACOMPRA_STD_R36` | Desvío estándar de los días entre compras, intervalos dentro de los últimos 1095 días. | `frecuencia_compra_std`(ventana='R36') |
| `MT_TICKETPROMEDIO_STD` | Desvío estándar de la venta por día de compra. Histórico. USD. | `ticket_promedio_std` |
| `MT_TICKETPROMEDIO_STD_R12` | Desvío estándar de la venta por día de compra en los últimos 365 días. USD. | `ticket_promedio_std`(ventana='R12') |
| `MT_TICKETPROMEDIO_STD_R24` | Desvío estándar de la venta por día de compra en los últimos 730 días. USD. | `ticket_promedio_std`(ventana='R24') |
| `MT_TICKETPROMEDIO_STD_R36` | Desvío estándar de la venta por día de compra en los últimos 1095 días. USD. | `ticket_promedio_std`(ventana='R36') |
| `MT_PROB_ACTIVO` | Probabilidad de que el cliente siga activo (modelo Pareto/NBD ajustado sobre todo el panel). | `prob_activo_col` |
| `BD_SEGMENTO_ACTIVIDAD` | Parámetros usados para MT_PROB_ACTIVO: GLOBAL (sin segmentación). | `segmento_actividad_col` |
| `MT_ANIO_INICIAL` | Año calendario de la primera compra. | `anio_inicial` |
| `MT_VENTA_ANIO_INICIAL` | Venta del año calendario de la primera compra, aunque sean pocos meses. USD. | `venta_anio_inicial` |
| `MT_ANIO_FINAL_CERRADO` | Último año calendario cerrado (anterior al año en curso) con compra. | `anio_final_cerrado` |
| `MT_VENTA_ANIO_FINAL_CERRADO` | Venta del último año calendario cerrado con compra. USD. | `venta_anio_final_cerrado` |
| `BD_MAXMENSUAL` | Mes calendario con mayor venta mensual promedio (ENERO..DICIEMBRE). | `max_mensual_nombre` |
| `MT_MAXMENSUAL` | Venta mensual promedio del mejor mes calendario. USD. | `max_mensual` |
| `BD_MINMENSUAL` | Mes calendario con menor venta mensual promedio (ENERO..DICIEMBRE). | `min_mensual_nombre` |
| `MT_MINMENSUAL` | Venta mensual promedio del peor mes calendario. USD. | `min_mensual` |
| `MT_IDDPORCENTUALMARGEN` | Pendiente de la recta del margen % mensual (sólo meses con venta positiva), expresada en gradianes (0 plano, 100 creciente, -100 decreciente). Positivo = más rentable. Histórico. | `idd_porcentual_margen` |
| `MT_IDDPORCENTUALMARGEN_R12` | Pendiente de la recta del margen % mensual de los últimos 12 meses cerrados, expresada en gradianes (0 plano, 100 creciente, -100 decreciente). | `idd_porcentual_margen`(ventana='R12') |
| `MT_IDDPORCENTUALMARGEN_R24` | Pendiente de la recta del margen % mensual de los últimos 24 meses cerrados, expresada en gradianes (0 plano, 100 creciente, -100 decreciente). | `idd_porcentual_margen`(ventana='R24') |
| `MT_IDDPORCENTUALMARGEN_R36` | Pendiente de la recta del margen % mensual de los últimos 36 meses cerrados, expresada en gradianes (0 plano, 100 creciente, -100 decreciente). | `idd_porcentual_margen`(ventana='R36') |
| `MT_PROM_ENERO` | Venta promedio de enero en los años de vida del cliente (meses cerrados, sin compra = 0). USD. | `promedio_mes`(mes=1) |
| `MT_PROM_FEBRERO` | Venta promedio de febrero en los años de vida del cliente (meses cerrados, sin compra = 0). USD. | `promedio_mes`(mes=2) |
| `MT_PROM_MARZO` | Venta promedio de marzo en los años de vida del cliente (meses cerrados, sin compra = 0). USD. | `promedio_mes`(mes=3) |
| `MT_PROM_ABRIL` | Venta promedio de abril en los años de vida del cliente (meses cerrados, sin compra = 0). USD. | `promedio_mes`(mes=4) |
| `MT_PROM_MAYO` | Venta promedio de mayo en los años de vida del cliente (meses cerrados, sin compra = 0). USD. | `promedio_mes`(mes=5) |
| `MT_PROM_JUNIO` | Venta promedio de junio en los años de vida del cliente (meses cerrados, sin compra = 0). USD. | `promedio_mes`(mes=6) |
| `MT_PROM_JULIO` | Venta promedio de julio en los años de vida del cliente (meses cerrados, sin compra = 0). USD. | `promedio_mes`(mes=7) |
| `MT_PROM_AGOSTO` | Venta promedio de agosto en los años de vida del cliente (meses cerrados, sin compra = 0). USD. | `promedio_mes`(mes=8) |
| `MT_PROM_SEPTIEMBRE` | Venta promedio de septiembre en los años de vida del cliente (meses cerrados, sin compra = 0). USD. | `promedio_mes`(mes=9) |
| `MT_PROM_OCTUBRE` | Venta promedio de octubre en los años de vida del cliente (meses cerrados, sin compra = 0). USD. | `promedio_mes`(mes=10) |
| `MT_PROM_NOVIEMBRE` | Venta promedio de noviembre en los años de vida del cliente (meses cerrados, sin compra = 0). USD. | `promedio_mes`(mes=11) |
| `MT_PROM_DICIEMBRE` | Venta promedio de diciembre en los años de vida del cliente (meses cerrados, sin compra = 0). USD. | `promedio_mes`(mes=12) |
| `MT_STD_ENERO` | Desvío estándar de la venta de enero en los años de vida del cliente. USD. | `std_mes`(mes=1) |
| `MT_STD_FEBRERO` | Desvío estándar de la venta de febrero en los años de vida del cliente. USD. | `std_mes`(mes=2) |
| `MT_STD_MARZO` | Desvío estándar de la venta de marzo en los años de vida del cliente. USD. | `std_mes`(mes=3) |
| `MT_STD_ABRIL` | Desvío estándar de la venta de abril en los años de vida del cliente. USD. | `std_mes`(mes=4) |
| `MT_STD_MAYO` | Desvío estándar de la venta de mayo en los años de vida del cliente. USD. | `std_mes`(mes=5) |
| `MT_STD_JUNIO` | Desvío estándar de la venta de junio en los años de vida del cliente. USD. | `std_mes`(mes=6) |
| `MT_STD_JULIO` | Desvío estándar de la venta de julio en los años de vida del cliente. USD. | `std_mes`(mes=7) |
| `MT_STD_AGOSTO` | Desvío estándar de la venta de agosto en los años de vida del cliente. USD. | `std_mes`(mes=8) |
| `MT_STD_SEPTIEMBRE` | Desvío estándar de la venta de septiembre en los años de vida del cliente. USD. | `std_mes`(mes=9) |
| `MT_STD_OCTUBRE` | Desvío estándar de la venta de octubre en los años de vida del cliente. USD. | `std_mes`(mes=10) |
| `MT_STD_NOVIEMBRE` | Desvío estándar de la venta de noviembre en los años de vida del cliente. USD. | `std_mes`(mes=11) |
| `MT_STD_DICIEMBRE` | Desvío estándar de la venta de diciembre en los años de vida del cliente. USD. | `std_mes`(mes=12) |
| `MT_VENTA_MES_ACTUAL` | Venta esperada del mes en curso: promedio histórico del mismo mes calendario. USD. | `venta_mes_actual_esperada` |
| `MT_VENTA_SIGUIENTE_MES` | Venta esperada del mes siguiente: promedio histórico de ese mes calendario. USD. | `venta_siguiente_mes_esperada` |
| `MT_VENTA_CUMPLIDA_ACTUAL` | Venta real del mes en curso, desde el día 1 hasta ayer. USD. | `venta_cumplida_actual` |
| `MT_PROB_INACTIVO` | Probabilidad de que el cliente esté inactivo: 1 - MT_PROB_ACTIVO (Pareto/NBD). | `prob_inactivo_col` |
| `BD_HISTORIAL` | JSON con la historia diaria: f0 = fecha de primera compra, d = días desde f0, v = venta USD, m = margen USD. | `historial_json` |
| `FECHA_CORTE` | Último día incluido en el cálculo (el día anterior a la ejecución). | `st_oracle.anotar` |
| `FECHA_DATOS_DESDE` | Primer día de la fuente contenido en la historia: desde cuándo hay datos. | `st_oracle.anotar` |
| `FECHA_RELECTURA_DESDE` | Desde qué día se releyó la fuente en la corrida que escribió la fila. En una corrida completa coincide con FECHA_DATOS_DESDE. | `st_oracle.anotar` |
| `TIPO_CORRIDA` | COMPLETA o INCREMENTAL. | `st_oracle.anotar` |
| `HUELLA_CONFIG` | Resumen de la configuración (categorías, SQL de la fuente, historial). Si cambia, la próxima corrida es completa. | `st_oracle.anotar` |